In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
# 1. 데이터 불러오기, # 2. 입력/정답 분리
digits = load_digits()
X = digits.data
y = digits.target

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1797, 64)
y shape: (1797,)


In [3]:
# 2. 정답 데이터 원-핫 인코딩
Y = pd.get_dummies(y).values

In [4]:
# 3. 학습 / 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=y)

In [5]:
# 4. 데이터 정규화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
# 5. 모델 생성
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)))
model.add(tf.keras.layers.Dense(32, activation="relu"))
model.add(tf.keras.layers.Dense(16, activation="relu"))
model.add(tf.keras.layers.Dense(10, activation="softmax"))
# 0~9까지 10개 클래스 분류하는 다중분류이므로 sofrmax활성화함수 사용
# X_train.shape[1]을 사용하면 입력 특성 개수가 바껴도 수정 필요 없음

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
# 6. 모델 설정
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
# 다중 분류이므로 categorical_crossentropy사용

In [8]:
# 7. 모델 구조 출력
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,938 (27.10 KB)

 Trainable params: 6,938 (27.10 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# 8. 모델 학습
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.3307 - loss: 2.0753 - val_accuracy: 0.5556 - val_loss: 1.7629
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7041 - loss: 1.3496 - val_accuracy: 0.7014 - val_loss: 1.1345
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8146 - loss: 0.7399 - val_accuracy: 0.8368 - val_loss: 0.6799
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9138 - loss: 0.3999 - val_accuracy: 0.8681 - val_loss: 0.4417
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9478 - loss: 0.2432 - val_accuracy: 0.9097 - val_loss: 0.3426
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9600 - loss: 0.1681 - val_accuracy: 0.9236 - val_loss: 0.2906
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9739 - loss: 0.1229 - val_accuracy: 0.9375 - val_loss: 0.2526
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9756 - loss: 0.0926 - val_accuracy: 0.9271 - val_

In [10]:
# 9. 테스트 데이터 예측
y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)
#10개 확률 중 가장 높은 확률을 가진 클래스의 인덱스(숫자)를 선택
y_test_labels = np.argmax(y_test, axis=1)
'''성능 평가를 위해 scikit-learn의 accuracy_score나 confusion_matrix 함수에 맞춰
원-핫 인코딩된 y_test도 1차원 배열로 변환'''

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


'성능 평가를 위해 scikit-learn의 accuracy_score나 confusion_matrix 함수에 맞춰\n원-핫 인코딩된 y_test도 1차원 배열로 변환'

In [11]:
# 10. 성능 평가
accuracy = accuracy_score(y_test_labels, y_pred)
print("\nAccuracy =", accuracy)
print("\nConfusion Matrix")
print(confusion_matrix(y_test_labels, y_pred))
print("\nClassification Report")
print(classification_report(y_test_labels, y_pred))


Accuracy = 0.9638888888888889

Confusion Matrix
[[35  0  1  0  0  0  0  0  0  0]
 [ 0 35  0  0  0  0  1  0  0  0]
 [ 0  1 34  0  0  0  0  0  0  0]
 [ 0  0  0 37  0  0  0  0  0  0]
 [ 0  0  0  0 35  0  0  1  0  0]
 [ 0  0  0  0  0 37  0  0  0  0]
 [ 0  0  0  0  0  0 35  0  1  0]
 [ 0  0  0  0  0  0  0 36  0  0]
 [ 0  4  0  1  0  0  0  1 29  0]
 [ 0  0  0  1  0  0  0  1  0 34]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.97      0.99        36
           1       0.88      0.97      0.92        36
           2       0.97      0.97      0.97        35
           3       0.95      1.00      0.97        37
           4       1.00      0.97      0.99        36
           5       1.00      1.00      1.00        37
           6       0.97      0.97      0.97        36
           7       0.92      1.00      0.96        36
           8       0.97      0.83      0.89        35
           9       1.00      0.94      0.97        36

 